In [1]:
# ==========================================
# 🛑 CELL 1: STACKED RECOGNIZER CLASS
# ==========================================
import numpy as np
from collections import Counter
from sklearn.neighbors import KNeighborsClassifier
from skimage.feature import hog

class StackedFaceRecognizer:
    def __init__(self, models_dict, train_feats, train_lbls, id_map):
        self.lbph = models_dict.get('LBPH')
        self.svm = models_dict.get('SVM')
        self.xgb = models_dict.get('XGB')
        self.le = models_dict.get('LE')
        self.X_train = np.array(train_feats)
        self.y_train = np.array(train_lbls)
        self.id_map = id_map
        
        # Internal memory to prevent duplicate counting
        self.session_history = {'SVM': set(), 'XGB': set(), 'LBPH': set()}

    def _reset_session(self):
        """Clears the short-term memory for a new frame."""
        self.session_history['SVM'].clear()
        self.session_history['XGB'].clear()
        self.session_history['LBPH'].clear()

    def _extract_hog(self, image):
        return hog(image, orientations=9, pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

    def _filter_best_candidates(self, all_faces):
        """
        INTERNAL TOURNAMENT:
        Filters duplicate predictions for the same person ID.
        Ensures LBPH keeps the Lowest distance, while SVM/XGB keep Highest confidence.
        """
        for model_name in ['SVM', 'XGB', 'LBPH']:
            # 1. Gather all candidates
            candidates = []
            for idx, face in enumerate(all_faces):
                p = face['preds'].get(model_name)
                if p: candidates.append((idx, p['id'], p['conf']))
            
            # 2. Group by ID (Normalized to String to prevent int/str duplications)
            grouped = {}
            for c in candidates:
                pid_key = str(c[1])  # CRITICAL FIX: "5" and 5 are now the same key
                if pid_key not in grouped: grouped[pid_key] = []
                grouped[pid_key].append(c)
                
            # 3. Filter
            for pid, items in grouped.items():
                if len(items) > 1:
                    # LBPH: Lower distance is better (False means Ascending sort)
                    # SVM/XGB: Higher probability is better (True means Descending sort)
                    is_higher_better = (model_name != 'LBPH')
                    
                    # Sort candidates
                    items.sort(key=lambda x: x[2], reverse=is_higher_better)
                    
                    # Winner is items[0]. Remove the losers.
                    for loser in items[1:]:
                        idx_rem = loser[0]
                        all_faces[idx_rem]['preds'][model_name] = None
                        
        return all_faces
    def _predict_single(self, face_img, preds):
        """INTERNAL: Calculates the final vote for ONE face."""
        
        # --- CONFIGURATION ---
        SVM_THRESHOLD = 0.4 
        CONF_THRESHOLD = 0.5   # 50% minimum for SVM/XGB
        LBPH_THRESHOLD = 75    # 75 maximum distance for LBPH
        # ---------------------

        svm_p = preds.get('SVM')
        xgb_p = preds.get('XGB')
        lbph_p = preds.get('LBPH')
        
        valid_votes = []
        valid_confs = []
        
        # 1. Collect Valid Votes (STRICT ENTRY REQUIREMENTS)
        
        # SVM GATEKEEPER
        if (svm_p and 
            svm_p['id'] not in self.session_history['SVM'] and 
            svm_p['conf'] >= SVM_THRESHOLD):  # <--- MUST BE >= 50%
            
            valid_votes.append(svm_p['id'])
            valid_confs.append(svm_p['conf'])
            self.session_history['SVM'].add(svm_p['id'])
            
        # XGB GATEKEEPER
        if (xgb_p and 
            xgb_p['id'] not in self.session_history['XGB'] and 
            xgb_p['conf'] >= CONF_THRESHOLD):  # <--- MUST BE >= 50%
            
            valid_votes.append(xgb_p['id'])
            valid_confs.append(xgb_p['conf'])
            self.session_history['XGB'].add(xgb_p['id'])

        # LBPH GATEKEEPER
        if (lbph_p and 
            lbph_p['id'] not in self.session_history['LBPH'] and 
            lbph_p['conf'] <= LBPH_THRESHOLD): # <--- MUST BE <= 75 DISTANCE
            
            valid_votes.append(lbph_p['id'])
            # Since we know it's <= 75, we treat it as valid. 
            # We give a high score (1.0) for very close matches (<50), else 0.6
            valid_confs.append(1.0 if lbph_p['conf'] < 50 else 0.6) 
            self.session_history['LBPH'].add(lbph_p['id'])
        
        # If everyone was filtered out, give up immediately
        if not valid_votes: 
            return None, 0.0, "Weak/No Votes"
        
        # 2. Consensus Logic
        top_vote, count = Counter(valid_votes).most_common(1)[0]
        
        # Strong Agreement (Majority Rule)
        if count >= 2: 
            return top_vote, 1.0, f"Voting ({count}/3)"
        
        # 3. Single Vote Preservation
        # Since we already filtered weak votes in Step 1, 
        # anyone remaining here is trusted enough to pass alone.
        if len(set(valid_votes)) == 1:
            idx = valid_votes.index(top_vote)
            return top_vote, valid_confs[idx], "Single Vote"

        # 4. Restricted KNN (Tie Breaker Only)
        unique_cands = list(set(valid_votes))
        hog_vec = self._extract_hog(face_img)
        mask = np.isin(self.y_train, unique_cands)
        
        if np.sum(mask) == 0: 
            return None, 0.0, "Err"
        
        n_neighbors = min(5, len(self.X_train[mask]))
        mini_knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric='euclidean', algorithm='brute')
        mini_knn.fit(self.X_train[mask], self.y_train[mask])
        
        final_id = mini_knn.predict([hog_vec])[0]
        final_conf = np.max(mini_knn.predict_proba([hog_vec])[0])
        
        return final_id, final_conf, "Restricted KNN"

    def process_frame(self, all_faces):
        """
        🛑 MAIN EXTERNAL CALL
        1. Resets session history.
        2. Filters duplicates across the whole frame (The Tournament).
        3. Predicts each face using the clean data.
        """
        self._reset_session()
        
        # Step A: Filter Duplicates
        cleaned_faces = self._filter_best_candidates(all_faces)
        
        final_results = []
        
        # Step B: Stack Predictions
        for face in cleaned_faces:
            pid, conf, method = self._predict_single(face['roi'], face['preds'])
            
            name = "Unknown"
            score_txt = ""
            
            if pid:
                name = self.id_map.get(pid, "Unknown")
                score_txt = f"{int(conf*100)}%"
            
            final_results.append({
                'bbox': face['bbox'], 
                'name': name, 
                'score': conf, 
                'score_txt': score_txt, 
                'info': method, 
                'roi': face['img_roi'],
                'raw_stats': face['raw_stats'],
                'raw_preds': face['unfiltered'] # Passed for GUI recovery if needed
            })
            
        return final_results

# Initialize logic


if 'models' in locals() and 'hog_feats' in locals():
    stacker = StackedFaceRecognizer(models, hog_feats, hog_lbls, id_to_name)
    print("✅ Stacker Class Loaded & Updated.")

In [2]:


# ==========================================
# 🛑 RUN THIS CELL FIRST (SETUP & LOADING)
# ==========================================
import joblib
import cv2
import xgboost as xgb
import numpy as np
import os
from skimage.feature import hog

# 1. Define the folder where images are (to get IDs)
DATA_PATH = r'..\Dataset\training\Cleaned_Training'

print("--- System Startup ---")

# --- A. Load Models ---
models = {}

try:
    models['SVM'] = joblib.load(r'..\models\svm_face_model.pkl')
    print("✅ SVM Loaded")
except: print("⚠️ SVM not found")

try:
    # Load XGBoost and its Label Encoder
    models['XGB'] = joblib.load(r'..\models\xgb_face_model.pkl')
    models['LE'] = joblib.load(r'..\models\label_encoder.pkl')
    print("✅ XGBoost Loaded")
except: print("⚠️ XGBoost or LabelEncoder not found")

try:
    lbph = cv2.face.LBPHFaceRecognizer_create()
    lbph.read(r'..\models\trainer.yml')
    models['LBPH'] = lbph
    print("✅ LBPH Loaded")
except: print("⚠️ LBPH not found")

# --- B. Create id_to_name Map ---
# This fixes the "name not defined" error. 
# It scans your folder and maps ID 1 -> "User 1", etc.
id_to_name = {}
if os.path.exists(DATA_PATH):
    img_files = os.listdir(DATA_PATH)
    unique_ids = set()
    for f in img_files:
        try:
            # Assumes format: User.1.jpg
            uid = int(f.split('.')[1])
            unique_ids.add(uid)
        except: pass
    
        # MANUAL NAME MAPPING
    id_to_name = {
        1: "Besheer",
        2: "Ashraf", 
        3: "Seif", 
        4: "Sallam", 
        5: "Roger", 
        6: "Omar"
    }
    print(f"✅ Loaded names for {len(id_to_name)} users.")
else:
    print("⚠️ Dataset path not found. Names will be 'Unknown'.")

# --- C. Initialize Stacker ---
# We need to reload training data briefly to initialize the Stacker class
try:
    hog_feats, hog_lbls = joblib.load(r'..\models\hog_features.pkl')
    if 'StackedFaceRecognizer' in globals():
        stacker = StackedFaceRecognizer(models, hog_feats, hog_lbls, id_to_name)
        print("✅ Stacker Initialized")
    else:
        print("❌ Error: Run the 'StackedFaceRecognizer' class cell first!")
except:
    print("⚠️ 'hog_features.pkl' missing. Stacker might fail.")

print("----------------------")
print("READY. Now run the GUI cell below.")





--- System Startup ---
✅ SVM Loaded
✅ XGBoost Loaded
✅ LBPH Loaded
✅ Loaded names for 6 users.
✅ Stacker Initialized
----------------------
READY. Now run the GUI cell below.


In [3]:
# ==========================================
# 🛑 CELL 2: GUI & DISPLAY
# ==========================================
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image as PILImage
import io
import copy

print("--- Launching GUI ---")

# Ensure names are loaded (Update this if your model output differs!)
id_to_name = { 
    1: "Besheer", 2: "Ashraf", 3: "Seif", 
    4: "Sallam", 5: "Roger", 6: "Omar" 
}

def compare_candidates(a, b):
    """
    Decides which face 'wins' the identity if two faces claim the same name.
    HIERARCHY:
    1. SVM (> 50%)
    2. LBPH (< 75 dist)
    3. XGBoost
    """

    stats_a = a['raw_stats']
    stats_b = b['raw_stats']
    # --- RULE 2: LBPH FALLBACK ---
    # If SVM was weak/equal, check LBPH (Lower is better)
    lbph_a = stats_a.get('LBPH', 999)
    lbph_b = stats_b.get('LBPH', 999)
    
    good_lbph_a = lbph_a < 75
    good_lbph_b = lbph_b < 75
    
    if good_lbph_a and not good_lbph_b: return a
    if good_lbph_b and not good_lbph_a: return b
    
    if good_lbph_a and good_lbph_b:
        return a if lbph_a < lbph_b else b # Lower distance wins
    # --- RULE 1: SVM SUPERIORITY ---
    # If one has a strong SVM and the other doesn't, the strong SVM wins.

    svm_a = stats_a.get('SVM', 0)
    svm_b = stats_b.get('SVM', 0)
    
    strong_a = svm_a > 0.40
    strong_b = svm_b > 0.40
    
    if strong_a and not strong_b: return a
    if strong_b and not strong_a: return b
    
    # If both have strong SVM, the higher confidence wins
    if strong_a and strong_b:
        return a if svm_a > svm_b else b
    # --- RULE 3: XGBOOST FINAL RESORT ---

    xgb_a = stats_a.get('XGB', 0)
    xgb_b = stats_b.get('XGB', 0)
    
    if xgb_a > xgb_b: return a
    if xgb_b > xgb_a: return b
        
    return a # Default to A if truly identical

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')

# Widgets
model_selector = widgets.ToggleButtons(
    options=['LBPH', 'SVM', 'XGB', 'Stacked'],
    description='Model:', button_style='success'
)
uploader = widgets.FileUpload(accept='image/*', multiple=False)
out_disp = widgets.Output()
out_table = widgets.Output()

def process_image(change):
    out_disp.clear_output(); out_table.clear_output()
    if not uploader.value: return
    try:
        if isinstance(uploader.value, tuple): f = uploader.value[0]
        else: f = next(iter(uploader.value.values()))
        img_np = np.array(PILImage.open(io.BytesIO(f['content'])))
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY) if len(img_np.shape) == 3 else img_np
    except: return

    detect_neighbors = 4 if model_selector.value != 'LBPH' else 3
    faces_rects = face_cascade.detectMultiScale(gray, 1.06, detect_neighbors, minSize=(49, 49))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    
    all_faces_raw = []

    # --- STEP 1: DETECTION & RAW EXTRACTION ---
    for i, (x, y, w, h) in enumerate(faces_rects):
        roi = cv2.resize(gray[y:y+h, x:x+w], (200, 200), interpolation=cv2.INTER_CUBIC)
        face_final = clahe.apply(cv2.bilateralFilter(roi, 5, 75, 75))
        
        preds = {'SVM': None, 'XGB': None, 'LBPH': None}
        raw_stats = {'SVM': 0.0, 'XGB': 0.0, 'LBPH': 999.0}
        
        # 1. SVM Prediction
        if 'SVM' in models:
            vec = stacker._extract_hog(face_final) if stacker else None
            try:
                prob = models['SVM'].predict_proba([vec])[0]; conf = np.max(prob)
                raw_stats['SVM'] = conf
                if conf > 0.40: 
                    pid = models['SVM'].classes_[np.argmax(prob)]
                    preds['SVM'] = {'id': pid, 'conf': conf}
            except: pass
            
        # 2. XGB Prediction
        if 'XGB' in models and 'LE' in models:
            vec = stacker._extract_hog(face_final) if stacker else None
            try:
                prob = models['XGB'].predict_proba([vec])[0]; conf = np.max(prob)
                raw_stats['XGB'] = conf
                if conf > 0.50:
                    decoded_id = models['LE'].inverse_transform([np.argmax(prob)])[0]
                    preds['XGB'] = {'id': decoded_id, 'conf': conf}
            except: pass

        # 3. LBPH Prediction
        if 'LBPH' in models:
            try:
                pid, dist = models['LBPH'].predict(face_final)
                raw_stats['LBPH'] = dist
                if dist < 75: preds['LBPH'] = {'id': pid, 'conf': dist}
            except: pass
        
        all_faces_raw.append({
            'roi': face_final, 'bbox': (x,y,w,h), 'preds': preds, 
            'unfiltered': copy.deepcopy(preds), 'raw_stats': raw_stats, 'img_roi': img_np[y:y+h, x:x+w]
        })

    # --- STEP 2: STACKER PROCESSING (The Clean Call) ---
    final_candidates = []
    
    if model_selector.value == 'Stacked' and stacker:
        # This one line handles filtering, duplicates, and stacking
        final_candidates = stacker.process_frame(all_faces_raw)
    
    else:
        # Standard Single Model Logic (Fallback)
        all_faces_raw = stacker._filter_best_candidates(all_faces_raw)
        for face in all_faces_raw:
            m = model_selector.value
            p = face['preds'].get(m)
            name = id_to_name.get(p['id'], "Unknown") if p else "Unknown"
            score_txt = f"{p['conf']*100:.1f}%" if (p and m!='LBPH') else (f"{p['conf']:.1f}" if p else "")
            final_candidates.append({
                'bbox': face['bbox'], 'name': name, 'score': 0, 'score_txt': score_txt,
                'info': m if p else "Filtered", 'roi': face['img_roi'], 
                'raw_stats': face['raw_stats'], 'raw_preds': face['unfiltered']
            })

    # --- STEP 3: FINAL GUI CLEANUP & RECOVERY ---
    # This ensures two different boxes don't claim to be "Sallam" on the screen
    grouped_final = {}
    for i, res in enumerate(final_candidates):
        if res['name'] == "Unknown": continue
        if res['name'] not in grouped_final: grouped_final[res['name']] = []
        grouped_final[res['name']].append(i)
    if model_selector.value == 'Stacked' and stacker:
        active_names = set(); losers_indices = []
        for name, indices in grouped_final.items():
            winner_idx = indices[0]
            if len(indices) > 1:
                for ch_idx in indices[1:]:
                    best = compare_candidates(final_candidates[winner_idx], final_candidates[ch_idx])
                    if best is final_candidates[ch_idx]: winner_idx = ch_idx
            active_names.add(name)
            for idx in indices:
                if idx != winner_idx: losers_indices.append(idx)
    
        # Attempt to recover losers using other models
            for idx in losers_indices:
                loser = final_candidates[idx]; preds = loser['raw_preds']; recovered = False
                
                # Try to find a non-duplicate identity in the secondary models
                for m_key in ['LBPH', 'SVM', 'XGB']:
                    if recovered: break
                    p = preds.get(m_key)
                    if p:
                        rec_name = id_to_name.get(p['id'], "Unknown")
                        is_valid = (m_key != 'LBPH' or p['conf'] < 75)
                        if is_valid and rec_name not in active_names:
                            score = f"{p['conf']*100:.1f}%" if m_key != 'LBPH' else f"{p['conf']:.1f}"
                            final_candidates[idx].update({'name': rec_name, 'info': f"Rec({m_key})", 'score_txt': score})
                            active_names.add(rec_name); recovered = True
                
                if not recovered:
                    final_candidates[idx].update({'name': "Unknown", 'info': "Duplicate", 'score_txt': ""})

    # --- STEP 4: DRAWING ---
    with out_disp:
        img_disp = img_np.copy()
        for res in final_candidates:
            x, y, w, h = res['bbox']
            color = (255, 0, 0) if res['name'] == "Unknown" else (0, 255, 0)
            label = f"{res['name']} ({res['score_txt']})" if res['name'] != "Unknown" else "Unknown"
            cv2.rectangle(img_disp, (x, y), (x+w, y+h), color, 2)
            cv2.putText(img_disp, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        display(PILImage.fromarray(img_disp))

    with out_table:
        rows = []
        for res in final_candidates:
            thumb = cv2.imencode('.png', cv2.cvtColor(res['roi'], cv2.COLOR_RGB2BGR))[1].tobytes()
            style = "color:green" if res['name'] != "Unknown" else "color:red"
            html = f"<b style='{style}'>{res['name']}</b><br><i>{res['info']}</i> {res['score_txt']}"
            rows.append(widgets.HBox([widgets.Image(value=thumb, format='png', width=50), widgets.HTML(html)]))
        display(widgets.VBox(rows))

uploader.observe(process_image, names='value')
model_selector.observe(process_image, names='value')
display(widgets.VBox([widgets.HTML("<h3>Face System v24 (Separate Cells)</h3>"), model_selector, uploader, out_disp, out_table]))

--- Launching GUI ---
